1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm
import seaborn as sns


2. Download NSE Data

In [60]:
tickers = pd.read_csv("tickers.csv")["Tickers"].tolist()
nse_data = pd.DataFrame()
for ticker in tickers:
    nse_data[ticker] = yf.download(ticker, start="2021-01-01", auto_adjust=False)["Adj Close"]


FileNotFoundError: [Errno 2] No such file or directory: 'tickers.csv'

3. Price Normalization Chart

In [ ]:
(nse_data//nse_data.iloc[0] * 100).plot(figsize=(12, 6))
plt.title("Stock Price Performance")
plt.show()

4. Return Calculation

In [ ]:
pf_returns = nse_data.pct_change().dropna()
print(pf_returns.head())

5. Annual Returns

In [ ]:
annual_returns = pf_returns.mean() * 252
print(annual_returns.head())
print(annual_returns.tail())


6. Covariance Matrix

In [ ]:
cov_matrix = pf_returns.cov() * 252
print(cov_matrix.head())

7. Correlation Heatmap

In [ ]:
corr_matrix = pf_returns.corr()

plt.figure(figsize=(12,8))

sns.heatmap(
    corr_matrix,
    annot=False,
    cmap="coolwarm"
)

plt.title("Stock Correlation Heatmap")
plt.show()

8. Monte Carlo Simulation

In [ ]:
portfolio_returns = []
portfolio_volatility = []
portfolio_weights = []

for i in range(10000):
    weights = np.random.random(len(tickers))
    weights /= np.sum(weights)

    portfolio_weights.append(weights)

    returns = np.dot(weights, annual_returns)
    portfolio_returns.append(returns)

    volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    portfolio_volatility.append(volatility)

portforlio_returns = np.array(portfolio_returns)
portfolio_volatility = np.array(portfolio_volatility)
portfolio_weights = np.array(portfolio_weights)

portfolio = pd.DataFrame({
    'Returns': portfolio_returns,
    'Volatility': portfolio_volatility
})

print(portfolio.head())

9. Sharpe Ratio Calculation

In [ ]:
risk_free_rate = 0.06

portfolio['Sharpe Ratio'] = (portfolio['Returns'] - risk_free_rate) / portfolio['Volatility']
print(portfolio.head())
print(portfolio.tail())

10. Best Portfolio

In [ ]:
max_sharpe_portfolio = portfolio.iloc[portfolio['Sharpe Ratio'].idxmax()]
min_volatility_portfolio = portfolio.iloc[portfolio['Volatility'].idxmin()]

print("Maximum Sharpe Ratio Portfolio:")
print(max_sharpe_portfolio)

print("Minimum Volatility Portfolio:")
print(min_volatility_portfolio)

In [ ]:
best_weights = portfolio_weights[portfolio['Sharpe Ratio'].idxmax()]
allocation = pd.DataFrame({
    'Stock': tickers,
    'Weight': best_weights
})
print(allocation)

11. Portfolio Allocation Pie Chart

In [ ]:
plt.figure(figsize=(10, 6))

plt.pie(
    allocation['Weight'],
    labels=allocation['Stock'],
    autopct='%1.1f%%'
    
)
plt.title("Optimal Portfolio Allocation")
plt.show()

12. Efficient Frontier Plot

In [ ]:
plt.figure(figsize=(12,6))

plt.scatter(
    portfolio["Volatility"],
    portfolio["Returns"],
    c = portfolio["Sharpe Ratio"],
    cmap="viridis",
    alpha=0.5
)

plt.colorbar(label="Sharpe Ratio")

plt.scatter(
    max_sharpe_portfolio["Volatility"],
    max_sharpe_portfolio["Returns"],
    color="red",
    s=200,
    label="Max Sharpe"
)

plt.scatter(
    min_volatility_portfolio["Volatility"],
    min_volatility_portfolio["Returns"],
    color="blue",
    s=200,
    label="Min Volatility"
)

plt.xlabel("Expected Volatility")
plt.ylabel("Expected Returns")
plt.title("Efficient Frontier")

plt.legend()
plt.show()